# Final project

Team1: Karl Prokop and Amanda Christianson

## Checkpoint 1: Sentinel-2 data selection and retrieval 

### Importing libraries and configuration of OAuth2 Client Credentials

In [1]:
# Importing libraries
import requests
import os
import json
from datetime import datetime
import zipfile
from pathlib import Path

# For visualization later
import matplotlib.pyplot as plt

In [5]:
# Configuration of credentials

# Direct assignment
COPERNICUS_CLIENT_ID = os.getenv('COPERNICUS_CLIENT_ID', 'sh-c5dcc309-63e8-491b-8c97-47925cbe91ea')
COPERNICUS_CLIENT_SECRET = os.getenv('COPERNICUS_CLIENT_SECRET', 'uZ0NKF3IlVgFdGQUbSOYlLDrdbjudxtL')

# Copernicus Dataspace API endpoints
AUTH_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SEARCH_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
DOWNLOAD_URL = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"

print("✓ Credentials configured")

✓ Credentials configured


In [6]:
def get_access_token(client_id, client_secret):
    """
    Get OAuth2 access token from Copernicus Dataspace using Client Credentials flow.
    
    This is the recommended method for server-to-server authentication and HPC jobs.
    
    Parameters:
    -----------
    client_id : str
        OAuth2 Client ID (starts with 'sh-')
    client_secret : str
        OAuth2 Client Secret
    
    Returns:
    --------
    str : Access token if successful, None otherwise
    """
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }
    
    try:
        response = requests.post(AUTH_URL, data=data, timeout=30)
        response.raise_for_status()
        token_data = response.json()
        
        # Extract token and expiration
        access_token = token_data["access_token"]
        expires_in = token_data.get("expires_in", 3600)
        
        print(f"✓ Token obtained (valid for {expires_in//60} minutes)")
        return access_token
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            print("❌ Authentication failed: Invalid credentials")
            print("   ✗ Check your CLIENT_ID and CLIENT_SECRET")
            print("   ✗ CLIENT_ID should start with 'sh-'")
        else:
            print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
        return None
        
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        return None

# Get access token
print("Authenticating with Copernicus Dataspace...")
access_token = get_access_token(COPERNICUS_CLIENT_ID, COPERNICUS_CLIENT_SECRET)

if access_token:
    print("✓ Successfully authenticated")
    headers = {"Authorization": f"Bearer {access_token}"}
else:
    print("❌ Authentication failed.")
    print("\nTroubleshooting:")
    print("1. Check environment variables are set:")
    print(f"   COPERNICUS_CLIENT_ID = {COPERNICUS_CLIENT_ID[:15]}...")
    print(f"   COPERNICUS_CLIENT_SECRET = {COPERNICUS_CLIENT_SECRET[:15]}...")
    print("2. Verify credentials in Copernicus Dashboard")
    print("3. See COPERNICUS_SETUP.md for detailed instructions")
    headers = None

Authenticating with Copernicus Dataspace...
✓ Token obtained (valid for 30 minutes)
✓ Successfully authenticated


### Define Search Parameters (region of interest and date range)

In [17]:
# Define your Region of Interest (ROI) as a bounding box
# Format: POLYGON((lon lat, lon lat, ...))
# Note: You will be selecting an MGRS tile within this region

# Bounding box coordinates [min_lon, min_lat, max_lon, max_lat]
# Middle of Sweden (Uppsala to Umeå), Sweden
min_lon, min_lat = 12.5, 60
max_lon, max_lat = 17.5, 63.5

# Create WKT POLYGON for API query
roi_polygon = f"POLYGON(({min_lon} {min_lat},{max_lon} {min_lat},{max_lon} {max_lat},{min_lon} {max_lat},{min_lon} {min_lat}))"

# Define date range
start_date = '2018-03-01T00:00:00.000Z'
end_date = '2018-10-31T23:59:59.999Z'

# Maximum cloud cover percentage (30% to get good data availability)
max_cloud_cover = 30

print(f"Search Parameters:")
print(f"  Region: Bavarian region (Central Europe with CORINE coverage)")
print(f"  Bounding Box: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")
print(f"  Date Range: {start_date[:10]} to {end_date[:10]}")
print(f"  Max Cloud Cover: {max_cloud_cover}%")
print(f"\nNote: Sentinel-2 divides the globe into MGRS tiles (100×100 km each).")
print(f"Your search will find all tiles intersecting this region.")

Search Parameters:
  Region: Bavarian region (Central Europe with CORINE coverage)
  Bounding Box: (12.5, 60) to (17.5, 63.5)
  Date Range: 2018-03-01 to 2018-10-31
  Max Cloud Cover: 30%

Note: Sentinel-2 divides the globe into MGRS tiles (100×100 km each).
Your search will find all tiles intersecting this region.


## Search Sentinel-2 Collection

Query the Copernicus Dataspace catalog for Sentinel-2 Level 2A imagery matching our criteria.

In [18]:
def search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover):
    """
    Search for Sentinel-2 L2A products in Copernicus Dataspace
    """
    # Build OData filter query
    filters = [
        f"Collection/Name eq 'SENTINEL-2'",
        f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A')",
        f"ContentDate/Start gt {start_date}",
        f"ContentDate/Start lt {end_date}",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{roi_polygon}')",
        f"Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt {max_cloud_cover})"
    ]
    
    filter_query = " and ".join(filters)
    
    params = {
        "$filter": filter_query,
        "$orderby": "ContentDate/Start asc",
        "$top": 1000  # Increased to get full date range across all tiles
    }
    
    try:
        response = requests.get(SEARCH_URL, params=params, timeout=60)
        response.raise_for_status()
        results = response.json()
        return results.get('value', [])
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return []

print("✓ Search function defined")

✓ Search function defined


In [19]:
print("Searching for Sentinel-2 products...")
products = search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover)

print(f"\n✓ Found {len(products)} Sentinel-2 L2A products")
print(f"✓ All products have <{max_cloud_cover}% cloud cover (filtered server-side)")
print(f"\nThese products span multiple MGRS tiles over your region.")
print(f"Select one MGRS tile and download ~4 acquisitions.\n")
print(f"First 5 products:")
for i, product in enumerate(products[:5]):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    
    print(f"  {i+1}. {name}")
    print(f"     Date: {date}, Size: {size:.2f} GB")

Searching for Sentinel-2 products...

✓ Found 1000 Sentinel-2 L2A products
✓ All products have <30% cloud cover (filtered server-side)

These products span multiple MGRS tiles over your region.
Select one MGRS tile and download ~4 acquisitions.

First 5 products:
  1. S2B_MSIL2A_20180301T103019_N0500_R108_T33VUK_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.20 GB
  2. S2B_MSIL2A_20180301T103019_N0500_R108_T32VPQ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.12 GB
  3. S2B_MSIL2A_20180301T103019_N0500_R108_T33VWJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.18 GB
  4. S2B_MSIL2A_20180301T103019_N0500_R108_T33VXL_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.17 GB
  5. S2B_MSIL2A_20180301T103019_N0500_R108_T33VVJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.16 GB


## Group Products by MGRS Tile

Organize products by MGRS tile to ensure we download multiple acquisitions of the same tile.

In [20]:
# Group products by MGRS tile
from collections import defaultdict

tiles = defaultdict(list)
for product in products:
    product_name = product.get('Name', '')
    # Extract MGRS tile from product name (e.g., T32UPD from S2A_MSIL2A_..._T32UPD_...)
    tile_id = product_name.split('_')[5] if len(product_name.split('_')) > 5 else 'Unknown'
    tiles[tile_id].append(product)

# Display available tiles and their acquisition counts
print("Available MGRS Tiles and Acquisition Counts:")
print("=" * 50)
for tile_id, tile_products in sorted(tiles.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\nTile {tile_id}: {len(tile_products)} acquisitions")
    for i, product in enumerate(tile_products[:5]):  # Show first 5
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. {date} - {size:.2f} GB")
    if len(tile_products) > 5:
        print(f"  ... and {len(tile_products) - 5} more")

print("\n" + "=" * 50)
print(f"\nRecommendation: Choose a tile with 4+ acquisitions for training data diversity.")

Available MGRS Tiles and Acquisition Counts:

Tile T33VWH: 58 acquisitions
  1. 2018-03-14 - 0.10 GB
  2. 2018-03-15 - 0.17 GB
  3. 2018-03-16 - 1.16 GB
  4. 2018-03-18 - 1.18 GB
  5. 2018-03-20 - 0.18 GB
  ... and 53 more

Tile T34VCN: 48 acquisitions
  1. 2018-03-18 - 0.95 GB
  2. 2018-03-20 - 0.94 GB
  3. 2018-03-21 - 0.12 GB
  4. 2018-03-25 - 0.84 GB
  5. 2018-03-26 - 0.10 GB
  ... and 43 more

Tile T34VCP: 46 acquisitions
  1. 2018-03-01 - 0.51 GB
  2. 2018-03-11 - 0.61 GB
  3. 2018-03-20 - 0.79 GB
  4. 2018-03-25 - 0.70 GB
  5. 2018-03-26 - 0.46 GB
  ... and 41 more

Tile T33VXJ: 46 acquisitions
  1. 2018-03-01 - 0.73 GB
  2. 2018-03-11 - 0.82 GB
  3. 2018-03-20 - 0.69 GB
  4. 2018-03-21 - 0.74 GB
  5. 2018-03-25 - 0.60 GB
  ... and 41 more

Tile T33VXH: 45 acquisitions
  1. 2018-03-18 - 1.08 GB
  2. 2018-03-20 - 1.02 GB
  3. 2018-03-21 - 0.43 GB
  4. 2018-03-25 - 0.98 GB
  5. 2018-03-28 - 1.01 GB
  ... and 40 more

Tile T33VWG: 43 acquisitions
  1. 2018-03-20 - 0.53 GB
  2. 2018

In [21]:
# Group products by MGRS tile
from collections import defaultdict

tiles = defaultdict(list)
for product in products:
    product_name = product.get('Name', '')
    # Extract MGRS tile from product name (e.g., T32UPD from S2A_MSIL2A_..._T32UPD_...)
    tile_id = product_name.split('_')[5] if len(product_name.split('_')) > 5 else 'Unknown'
    tiles[tile_id].append(product)

# Display available tiles and their acquisition counts
print("Available MGRS Tiles and Acquisition Counts:")
print("=" * 50)
for tile_id, tile_products in sorted(tiles.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\nTile {tile_id}: {len(tile_products)} acquisitions")
    for i, product in enumerate(tile_products[:5]):  # Show first 5
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. {date} - {size:.2f} GB")
    if len(tile_products) > 5:
        print(f"  ... and {len(tile_products) - 5} more")

print("\n" + "=" * 50)
print(f"\nRecommendation: Choose a tile with 4+ acquisitions for training data diversity.")
# Select a tile to work with (choose the one with most acquisitions, or specify manually)
# Option 1: Automatic - select tile with most acquisitions
selected_tile = max(tiles.items(), key=lambda x: len(x[1]))[0] if tiles else None

# Option 2: Manual selection - uncomment and specify tile ID
# selected_tile = "T32UPD"  # Replace with your chosen tile

if selected_tile:
    tile_products = tiles[selected_tile]
    num_acquisitions = len(tile_products)
    
    print(f"Selected MGRS Tile: {selected_tile}")
    print(f"Total acquisitions available: {num_acquisitions}")
    
    # Select 4 evenly spaced acquisitions for temporal diversity
    num_to_select = 4
    if num_acquisitions >= num_to_select:
        # Calculate indices for evenly spaced selection
        indices = [int(i * (num_acquisitions - 1) / (num_to_select - 1)) for i in range(num_to_select)]
        selected_products = [tile_products[i] for i in indices]
    else:
        # If fewer than 4 acquisitions, use all of them
        selected_products = tile_products
        indices = list(range(len(tile_products)))
    
    print(f"\nSelected {len(selected_products)} evenly-spaced acquisitions for temporal diversity:")
    print("=" * 70)
    for i, (idx, product) in enumerate(zip(indices, selected_products)):
        name = product.get('Name', 'Unknown')
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. [{idx+1}/{num_acquisitions}] {date} - {size:.2f} GB")
        print(f"      {name}")
    print("=" * 70)
    print("\nThese acquisitions span the full date range for better training data diversity.")
else:
    print("❌ No tiles found. Adjust your search parameters.")
    selected_products = []

Available MGRS Tiles and Acquisition Counts:

Tile T33VWH: 58 acquisitions
  1. 2018-03-14 - 0.10 GB
  2. 2018-03-15 - 0.17 GB
  3. 2018-03-16 - 1.16 GB
  4. 2018-03-18 - 1.18 GB
  5. 2018-03-20 - 0.18 GB
  ... and 53 more

Tile T34VCN: 48 acquisitions
  1. 2018-03-18 - 0.95 GB
  2. 2018-03-20 - 0.94 GB
  3. 2018-03-21 - 0.12 GB
  4. 2018-03-25 - 0.84 GB
  5. 2018-03-26 - 0.10 GB
  ... and 43 more

Tile T34VCP: 46 acquisitions
  1. 2018-03-01 - 0.51 GB
  2. 2018-03-11 - 0.61 GB
  3. 2018-03-20 - 0.79 GB
  4. 2018-03-25 - 0.70 GB
  5. 2018-03-26 - 0.46 GB
  ... and 41 more

Tile T33VXJ: 46 acquisitions
  1. 2018-03-01 - 0.73 GB
  2. 2018-03-11 - 0.82 GB
  3. 2018-03-20 - 0.69 GB
  4. 2018-03-21 - 0.74 GB
  5. 2018-03-25 - 0.60 GB
  ... and 41 more

Tile T33VXH: 45 acquisitions
  1. 2018-03-18 - 1.08 GB
  2. 2018-03-20 - 1.02 GB
  3. 2018-03-21 - 0.43 GB
  4. 2018-03-25 - 0.98 GB
  5. 2018-03-28 - 1.01 GB
  ... and 40 more

Tile T33VWG: 43 acquisitions
  1. 2018-03-20 - 0.53 GB
  2. 2018